In [1]:
import numpy as np

from qmcpy.discrete_distribution import IIDStdUniform, Lattice
from qmcpy.true_measure import Uniform, AcceptReject

# --- Helper PDFs ---------------------------------------------------------

def beta_2_5_pdf(x: np.ndarray) -> np.ndarray:
    """
    Beta(2,5) density on [0,1].

    f(x) = 42 x^(2-1) (1-x)^(5-1) on [0,1], 0 otherwise
    """
    x = np.asarray(x).reshape(-1)
    a, b = 2.0, 5.0
    coeff = 42.0  # 1 / Beta(2,5) = 42
    f = coeff * np.power(x, a - 1.0) * np.power(1.0 - x, b - 1.0)
    f[(x < 0.0) | (x > 1.0)] = 0.0
    return f

def uniform_01_pdf(x: np.ndarray) -> np.ndarray:
    """
    Uniform(0,1) density.

    g(x) = 1 on [0,1], 0 otherwise.
    """
    x = np.asarray(x).reshape(-1)
    g = np.ones_like(x)
    g[(x < 0.0) | (x > 1.0)] = 0.0
    return g


In [ ]:
# --- Test 1: IID driver sanity test -------------------------------------

print("=== IIDSanityTest ===")

# Target / proposal dimension
d = 1

# IID driver for the proposal
dd_prop = IIDStdUniform(d)

# Proposal = Uniform(0,1)
proposal = Uniform(dd_prop, lower_bound=0.0, upper_bound=1.0)

# IID driver for acceptance–rejection
dd_driver = IIDStdUniform(d + 1)

# Bound c: needs to satisfy f(x) <= c g(x); for Beta(2,5) max f is ~2.4576
c_bound = 3.0

ar = AcceptReject(
    proposal_measure=proposal,
    target_pdf=beta_2_5_pdf,
    proposal_pdf=uniform_01_pdf,
    bound_c=c_bound,
    driver_discrete_distrib=dd_driver,
)

# request 5000 accepted samples
n = 5000
samples = ar(n=n)

# Checks
print("samples.shape:", samples.shape)

sample_mean = samples.mean()
true_mean = 2.0 / 7.0 # for beta(2,5) distribution mean = 2 / (2 + 5)

print("sample mean =", sample_mean)
print("true mean   =", true_mean)
print("abs error   =", abs(sample_mean - true_mean))

# A loose sanity threshold: 0.01 (MC standard error here is ~0.002)
if abs(sample_mean - true_mean) < 0.01:
    print("PASS: IID sanity test looks good.")
else:
    print("WARNING: IID test mean too far from truth.")


=== IIDSanityTest ===
samples.shape: (5000, 1)
sample mean = 0.28820194111555486
true mean   = 0.2857142857142857
abs error   = 0.002487655401269162
PASS: IID sanity test looks good.


In [5]:
# --- Test 2: Lattice driver test (LD) -----------------------------------

print("=== LatticeTest ===")

from qmcpy.discrete_distribution import Lattice

# Target / proposal dimension
d = 1

# Lattice driver for the proposal (2D)
dd_prop_lat = Lattice(dimension=d)

# Proposal = Uniform(0,1) driven by lattice
proposal_lat = Uniform(dd_prop_lat, lower_bound=0.0, upper_bound=1.0)

# Lattice driver for acceptance–rejection in 2D: (u, v)
#dd_driver_lat = Lattice(dimension=d + 1)
dd_driver_lat = Lattice(dimension=d+1, order="GRAY")

# Beta(2,5) parameters (same target as IID test)
a, b = 2.0, 5.0

# Bound c: must satisfy f(x) <= c g(x); for Beta(2,5) max f ≈ 2.46, so c = 3.0
c_bound_lat = 3.0

ar_lat = AcceptReject(
    proposal_measure=proposal_lat,
    target_pdf=beta_2_5_pdf,
    proposal_pdf=uniform_01_pdf,
    bound_c=c_bound_lat,
    driver_discrete_distrib=dd_driver_lat,
)

# Use a power-of-two n (nice for QMC / lattice analysis, though not required)
n_lat = 4096
samples_lat = ar_lat(n=n_lat)

print("samples_lat.shape:", samples_lat.shape)

sample_mean_lat = samples_lat.mean()
true_mean_lat = a / (a + b)

print("lattice sample mean =", sample_mean_lat)
print("true mean          =", true_mean_lat)
print("abs error          =", abs(sample_mean_lat - true_mean_lat))

# Theoretical variance of Beta(a,b)
var_lat = (a * b) / ((a + b) ** 2 * (a + b + 1))
se_lat = np.sqrt(var_lat / n_lat)

print("theoretical MC SE   =", se_lat)

# Loose sanity threshold (5 standard errors) – QMC should usually be better
if abs(sample_mean_lat - true_mean_lat) < 5 * se_lat:
    print("PASS: Lattice test is within expected error band.")
else:
    print("WARNING: Lattice test mean outside expected range.")


=== LatticeTest ===
samples_lat.shape: (4096, 1)
lattice sample mean = 0.28915541056232386
true mean          = 0.2857142857142857
abs error          = 0.0034411248480381573
theoretical MC SE   = 0.0024956115820310154
PASS: Lattice test is within expected error band.


In [6]:
# --- Test 3: Halton driver test (LD) -----------------------------------

print("=== HaltonTest ===")

from qmcpy.discrete_distribution import Halton

# Target / proposal dimension
d = 1

# Halton driver for the proposal (1D)
dd_prop_hal = Halton(dimension=d)

# Proposal = Uniform(0,1) driven by Halton
proposal_hal = Uniform(dd_prop_hal, lower_bound=0.0, upper_bound=1.0)

# Halton driver for acceptance–rejection in 2D: (u, v)
dd_driver_hal = Halton(dimension=d + 1)

# Beta(2,5) parameters (same target)
a, b = 2.0, 5.0

# Bound c (same reasoning as before: c >= sup f/g)
c_bound_hal = 3.0

ar_hal = AcceptReject(
    proposal_measure=proposal_hal,
    target_pdf=beta_2_5_pdf,
    proposal_pdf=uniform_01_pdf,
    bound_c=c_bound_hal,
    driver_discrete_distrib=dd_driver_hal,
)

# n does not have to be a power of 2 for Halton, but we can keep it for consistency
n_hal = 4096
samples_hal = ar_hal(n=n_hal)

print("samples_hal.shape:", samples_hal.shape)

sample_mean_hal = samples_hal.mean()
true_mean_hal = a / (a + b)

print("halton sample mean =", sample_mean_hal)
print("true mean         =", true_mean_hal)
print("abs error         =", abs(sample_mean_hal - true_mean_hal))

# Theoretical variance & MC SE for reference
var_hal = (a * b) / ((a + b) ** 2 * (a + b + 1))
se_hal = np.sqrt(var_hal / n_hal)

print("theoretical MC SE  =", se_hal)

# Sanity check: error within 5 * MC SE
if abs(sample_mean_hal - true_mean_hal) < 5 * se_hal:
    print("PASS: Halton test is within expected error band.")
else:
    print("WARNING: Halton test mean outside expected range.")


=== HaltonTest ===
samples_hal.shape: (4096, 1)
halton sample mean = 0.28882361783938665
true mean         = 0.2857142857142857
abs error         = 0.00310933212510095
theoretical MC SE  = 0.0024956115820310154
PASS: Halton test is within expected error band.
